# ********************************************
### IMPORT ALL THE LIBRARIES
# ********************************************

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path
import random
import warnings
warnings.filterwarnings('ignore') 

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
np.random.seed(42)
random.seed(42)

# ── TABLE 1: CAMPAIGNS ──────────────────────────────────────────

In [3]:
n_campaigns = 800

channels = ['Google Ads', 'TV Channels', 'Email Marketing', 'SEO', 'YouTube Ads', 'Influencer', 'Affiliate']

campaign_types = ['Brand Awareness', 'Lead Generation', 'Retargeting', 'Conversion', 'Engagement'] 

regions = ['North', 'South', 'East', 'West', 'Central'] 

product_categories = ['Electronics', 'Fashion', 'Beauty & Skincare', 'Home & Living', 'Food & Beverages', 'Health & Fitness', 'Travel', 'Education'] 

clients = [f'Client_{i:03d}' for i in range(1, 51)] 

campaign_status = ['Active', 'Completed', 'Paused'] 

start_dates = [datetime(2022, 1, 1) + timedelta(days=random.randint(0, 365)) 
               for _ in range(n_campaigns)]

end_dates = [sd + timedelta(days=random.randint(200, 400)) 
             for sd in start_dates]

budgets = np.random.choice(
    [np.random.randint(50000, 200000),    # Small
     np.random.randint(200000, 800000),   # Medium  
     np.random.randint(800000, 3000000)], # Large
    n_campaigns,
    p=[0.5, 0.35, 0.15]
)

In [4]:
df_campaigns = pd.DataFrame({
    'Campaign_ID':       [f'CMP_{i:04d}' for i in range(1, n_campaigns + 1)],
    'Campaign_Name':     [f'{random.choice(product_categories)}_{random.choice(campaign_types)}_Campaign_{i:04d}' 
                          for i in range(1, n_campaigns + 1)],
    'Channel':           np.random.choice(channels, n_campaigns, 
                          p=[0.22, 0.20, 0.15, 0.10, 0.13, 0.10, 0.10]),
    'Campaign_Type':     np.random.choice(campaign_types, n_campaigns),
    'Target_Audience':   np.random.choice(['18-24', '25-34', '35-44', '45-54', '55+'], n_campaigns),
    'Region':            np.random.choice(regions, n_campaigns),
    'Start_Date':        start_dates,
    'End_Date':          end_dates,
    'Total_Budget':      budgets,
    'Product_Category':  np.random.choice(product_categories, n_campaigns),
    'Client_Name':       np.random.choice(clients, n_campaigns),
    'Campaign_Status':   np.random.choice(campaign_status, n_campaigns, 
                          p=[0.30, 0.60, 0.10])
})

In [5]:
print(f"Campaigns Table Shape: {df_campaigns.shape}")
print(f"\nFirst 3 rows:")
df_campaigns.head(3)

Campaigns Table Shape: (800, 12)

First 3 rows:


,Campaign_ID,Campaign_Name,Channel,Campaign_Type,Target_Audience,Region,Start_Date,End_Date,Total_Budget,Product_Category,Client_Name,Campaign_Status
0,CMP_0001,Home & Living_Conversion_Campaign_0001,SEO,Brand Awareness,55+,North,2022-11-24,2023-09-30,331932,Home & Living,Client_032,Active
1,CMP_0002,Education_Lead Generation_Campaign_0002,Email Marketing,Conversion,35-44,North,2022-02-27,2022-12-16,171958,Food & Beverages,Client_042,Completed
2,CMP_0003,Travel_Retargeting_Campaign_0003,YouTube Ads,Engagement,45-54,South,2022-01-13,2023-01-10,171958,Fashion,Client_018,Active


# ── TABLE 2: CAMPAIGN PERFORMANCE ───────────────────────────────

In [7]:

records = []

for _, campaign in df_campaigns.iterrows():
    
    cmp_id   = campaign['Campaign_ID']
    channel  = campaign['Channel']
    start    = campaign['Start_Date']
    end      = campaign['End_Date']
    budget   = campaign['Total_Budget']
    
   
    date_range = pd.date_range(start=start, end=end, freq='D')
    n_days = len(date_range)
    
    # Channel wise realistic ROAS ranges
    roas_range = {
        'Google Ads':       (3.5, 7.0),
        'TV Channels':         (2.5, 5.5),
        'Email Marketing':  (8.0, 15.0),
        'SEO':              (5.0, 12.0),
        'YouTube Ads':      (2.0, 4.5),
        'Influencer':       (1.5, 4.0),
        'Affiliate':        (4.0, 9.0)
    }
    

    ctr_range = {
        'Google Ads':       (0.03, 0.08),
        'TV Channels':         (0.02, 0.06),
        'Email Marketing':  (0.15, 0.35),
        'SEO':              (0.05, 0.12),
        'YouTube Ads':      (0.01, 0.04),
        'Influencer':       (0.02, 0.07),
        'Affiliate':        (0.04, 0.10)
    }
    
    daily_budget = budget / n_days
    
    for date in date_range:
        
        spend        = daily_budget * np.random.uniform(0.7, 1.3)
        impressions  = int(spend * np.random.uniform(80, 200))
        ctr          = np.random.uniform(*ctr_range[channel])
        clicks       = int(impressions * ctr)
        cvr          = np.random.uniform(0.01, 0.08)
        conversions  = int(clicks * cvr)
        roas         = np.random.uniform(*roas_range[channel])
        revenue      = spend * roas
        cpc          = spend / clicks if clicks > 0 else 0
        cpa          = spend / conversions if conversions > 0 else 0
        
        records.append({
            'Performance_ID':       f'PERF_{len(records)+1:07d}',
            'Campaign_ID':           cmp_id,
            'Date':                  date,
            'Impressions':           impressions,
            'Clicks':                clicks,
            'Conversions':           conversions,
            'Spend':                 round(spend, 2),
            'Revenue':               round(revenue, 2),
            'CTR':                   round(ctr * 100, 2),
            'CVR':                   round(cvr * 100, 2),
            'CPC':                   round(cpc, 2),
            'CPA':                   round(cpa, 2),
            'ROAS':                  round(roas, 2),
            'Bounce_Rate':           round(np.random.uniform(30, 75), 2),
            'Avg_Session_Duration':  round(np.random.uniform(60, 480), 0)
        })

df_performance = pd.DataFrame(records)

print(f"Performance Table Shape: {df_performance.shape}")
print(f"\nFirst 3 rows:")
df_performance.head(3)

Performance Table Shape: (242487, 15)

First 3 rows:


,Performance_ID,Campaign_ID,Date,Impressions,Clicks,Conversions,Spend,Revenue,CTR,CVR,CPC,CPA,ROAS,Bounce_Rate,Avg_Session_Duration
0,PERF_0000001,CMP_0001,2022-11-24,224203,14885,566,1233.87,7055.89,6.64,3.81,0.08,2.18,5.72,72.34,180.0
1,PERF_0000002,CMP_0001,2022-11-25,110921,11779,271,1128.79,12039.31,10.62,2.30,0.10,4.17,10.67,48.08,303.0
2,PERF_0000003,CMP_0001,2022-11-26,171027,17648,614,919.83,8368.21,10.32,3.48,0.05,1.50,9.10,35.80,127.0


# ── TABLE 3: INFLUENCER & AFFILIATE ─────────────────────────────

In [8]:
n_records = 25000

platforms = ['Instagram', 'YouTube', 'Facebook', 
             'Twitter', 'LinkedIn', 'Blog/Website']

partner_types = ['Influencer', 'Affiliate']

influencer_tiers = ['Nano (1K-10K)', 'Micro (10K-100K)', 
                    'Macro (100K-1M)', 'Mega (1M+)']

influencer_campaigns = df_campaigns[
    df_campaigns['Channel'].isin(['Influencer', 'Affiliate'])
]['Campaign_ID'].tolist()

all_campaign_ids = df_campaigns['Campaign_ID'].tolist()

partner_names = [f'Partner_{i:04d}' for i in range(1, 501)]

df_influencer = pd.DataFrame({
    'Record_ID':         [f'INF_{i:06d}' for i in range(1, n_records + 1)],
    'Campaign_ID':        np.random.choice(all_campaign_ids, n_records),
    'Partner_Type':       np.random.choice(partner_types, n_records, p=[0.6, 0.4]),
    'Partner_Name':       np.random.choice(partner_names, n_records),
    'Platform':           np.random.choice(platforms, n_records, 
                           p=[0.35, 0.25, 0.15, 0.10, 0.08, 0.07]),
    'Influencer_Tier':    np.random.choice(influencer_tiers, n_records,
                           p=[0.40, 0.35, 0.18, 0.07]),
    'Reach':              np.random.randint(1000, 5000000, n_records),
    'Engagement_Rate':    np.random.uniform(1.0, 8.0, n_records).round(2),
    'Clicks':             np.random.randint(100, 50000, n_records),
    'Conversions':        np.random.randint(10, 5000, n_records),
    'Commission_Paid':    np.random.uniform(500, 50000, n_records).round(2),
    'Revenue_Generated':  np.random.uniform(1000, 200000, n_records).round(2),
    'Post_Date':          [datetime(2022, 1, 1) + timedelta(days=random.randint(0, 1000)) 
                           for _ in range(n_records)],
})

print(f"Influencer/Affiliate Table Shape: {df_influencer.shape}")
print(f"\nFirst 3 rows:")
df_influencer.head(3)

Influencer/Affiliate Table Shape: (25000, 13)

First 3 rows:


,Record_ID,Campaign_ID,Partner_Type,Partner_Name,Platform,Influencer_Tier,Reach,Engagement_Rate,Clicks,Conversions,Commission_Paid,Revenue_Generated,Post_Date
0,INF_000001,CMP_0068,Influencer,Partner_0377,Instagram,Nano (1K-10K),2158370,7.55,8939,565,16694.52,138643.25,2024-01-19
1,INF_000002,CMP_0475,Affiliate,Partner_0084,YouTube,Nano (1K-10K),2498429,1.72,29848,2650,21208.74,44226.65,2024-09-22
2,INF_000003,CMP_0399,Influencer,Partner_0158,LinkedIn,Nano (1K-10K),1043006,1.92,24472,1855,31753.68,16045.32,2023-04-15


# ── SAVE ALL 3 TABLES AS CSV ─────────────────────────────────────

In [10]:
from pathlib import Path
dataset_path = Path("..") / "Dataset"
dataset_path.mkdir(parents=True, exist_ok=True)

df_campaigns.to_csv(dataset_path / "campaigns.csv", index=False)
df_performance.to_csv(dataset_path / "campaign_performance.csv", index=False)
df_influencer.to_csv(dataset_path / "influencer_affiliate.csv", index=False)

In [11]:
print(f"\n📊 Dataset Summary:")
print(f"campaigns.csv          → {df_campaigns.shape[0]:,} rows × {df_campaigns.shape[1]} columns")
print(f"campaign_performance.csv → {df_performance.shape[0]:,} rows × {df_performance.shape[1]} columns")
print(f"influencer_affiliate.csv → {df_influencer.shape[0]:,} rows × {df_influencer.shape[1]} columns")
print(f"\nTotal Records: {df_campaigns.shape[0] + df_performance.shape[0] + df_influencer.shape[0]:,}")



📊 Dataset Summary:
campaigns.csv          → 800 rows × 12 columns
campaign_performance.csv → 242,487 rows × 15 columns
influencer_affiliate.csv → 25,000 rows × 13 columns

Total Records: 268,287
